In [1]:
import sys
sys.path.append('..')

from utilities_amigocloud import AmigocloudFunctions
from sqlalchemy import create_engine, text
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import Point, Polygon
from shapely import wkb

from config import RUTA_UNIDAD_ONE_DRIVE
from config import RUTA_LOCAL_ONE_DRIVE
from config import API_AMIGOCLOUD_TOKEN_ADM
from config import POSTGRES_UTEA

In [67]:
# conexion a base de datos
def obtener_engine():
    return create_engine(
        f"postgresql+psycopg2://{POSTGRES_UTEA['USER']}:{POSTGRES_UTEA['PASSWORD']}@{POSTGRES_UTEA['HOST']}:{POSTGRES_UTEA['PORT']}/{POSTGRES_UTEA['DATABASE']}"
    )

# obtener datos de todo catastro
def get_catastro():
    engine = obtener_engine()
    try:
        query = f'''
            SELECT * FROM catastro_iag.catastro
        '''
        gdf = gpd.read_postgis(query, engine, geom_col='geom')
        return gdf
    except Exception as e:
        print(f"❌ Error al obtener la capa de catastro: {e}")
        return gpd.GeoDataFrame()
    return None

# eliminar todas las propiedades cuyo unidad_01 este en la lista recibida
def eliminar_props(lista_cod):
    cod_string = ", ".join(map(str, lista_cod))
    query = f"""DELETE FROM dataset_392292 WHERE unidad_01 in ({cod_string})"""
    select = amigocloud.ejecutar_query_sql(ID_PROYECTO, query, 'post')
    return select

# funcion auxiliar para convertir poligono en multipoligono
def convertir_a_multipolygon(geometry):
    if isinstance(geometry, Polygon):
        return MultiPolygon([geometry])
    return geometry

# convertir poligono en formato wkb
def convertir_a_wkb(polygon):
    wkb_data = wkb.dumps(polygon, hex=True)
    return wkb_data

# cargar lotes de geodataframe a amigocloud de forma masiva
def cargar_props_a_amigocloud_bulk(gdf):
    # 1. Preparación de datos y reproyección
    gdf_gral = gdf.to_crs(epsg=4326).copy()
    # Asegurar que usamos la columna de geometría correcta
    # Si tu columna se llama 'geom', la procesamos así:
    gdf_gral['geometry_processed'] = gdf_gral['geom'].apply(convertir_a_multipolygon)
    # --- FUNCIÓN AUXILIAR DE LIMPIEZA ---
    def sql_v(val, tipo='num'):
        """
        Convierte valores de Python/Pandas a sintaxis SQL válida.
        tipo: 'num', 'str', 'date'
        """
        if pd.isna(val) or val is None or str(val).lower() == 'nan':
            return "NULL"
        if tipo == 'str':
            # Quitar espacios extras y escapar comillas simples
            texto = str(val).strip().replace("'", "''")
            return f"'{texto}'"
        if tipo == 'date':
            return f"'{val}'"
            
        # Para números (int, float)
        return str(val)
    # 2. Construir la lista de valores para el SQL
    valores_rows = []
    for index, row in gdf_gral.iterrows():
        # Generar el WKB Hexadecimal
        wkb_hex = convertir_a_wkb(row['geometry_processed'])
        # Formateamos la fila usando la función sql_v
        # El orden debe ser idéntico al INSERT del paso 4
        fila_sql = f"""(
            ST_SetSRID(ST_GeomFromWKB('\\x{wkb_hex}'), 4326),
            {sql_v(row.get('idd'))},
            {sql_v(row.get('unidad_01'))},
            {sql_v(row.get('unidad_02'), 'str')},
            {sql_v(row.get('unidad_03'))},
            {sql_v(row.get('unidad_04'), 'str')},
            {sql_v(row.get('unidad_05'), 'str')},
            {sql_v(row.get('fs'), 'date')},
            {sql_v(row.get('area'))}
        )"""
        valores_rows.append(fila_sql)

    # 3. Unir todas las filas
    todos_los_valores = ",\n".join(valores_rows)
    # 4. Crear el Query Final
    insert_sql = f"""
        INSERT INTO dataset_392292
        (geometrias, idd, unidad_01, unidad_02, unidad_03, unidad_04, unidad_05, fs, area)
        VALUES 
        {todos_los_valores};
    """
    # 5. Ejecutar
    try:
        print(f"Enviando {len(gdf_gral)} registros a AmigoCloud (Dataset 390174)...")
        resultado = amigocloud.ejecutar_query_sql(ID_PROYECTO, insert_sql, 'post') 
        if resultado:
            print("✅ Carga masiva exitosa.")
            # Opcional: marcar como planificado localmente
            # for l_id in gdf_gral['idd']: marcar_lote_como_planificado(l_id)
        else:
            print("❌ El servidor no confirmó la carga.")
            
    except Exception as e:
        print(f"❌ Error en el Insert Masivo: {e}")
    return None

In [3]:
ID_PROYECTO = 36863
amigocloud = AmigocloudFunctions(token=API_AMIGOCLOUD_TOKEN_ADM)
amigocloud

In [49]:
# get todos los datos de catastro
gdf_catastro = get_catastro()

In [50]:
# lista de codigos de propiedad a eliminar
CAMANHA = 2026
lista_props = [
299,
30,
482,
2238,
2307
]

In [51]:
# eliminar propiedades
eliminar_props(lista_props)

{'query': 'DELETE FROM dataset_392292 WHERE unidad_01 in (299, 30, 482, 2238, 2307)',
 'count': 0}

In [55]:
# leccionar propiedades segun lista de codigos
props_select_segun_lista = gdf_catastro[gdf_catastro['unidad_01'].isin(lista_props)]

In [59]:
# La forma correcta y segura
props_select_segun_lista['fs'] = pd.to_datetime(props_select_segun_lista['fs'])

C:\Users\Usuario\anaconda3\envs\utea\lib\site-packages\geopandas\geodataframe.py:1525: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


In [60]:
df_2026 = props_select_segun_lista[props_select_segun_lista['fs'].dt.year == CAMANHA].copy()

In [68]:
# cargar gdf a amigocloud de forma masiva
cargar_props_a_amigocloud_bulk(df_2026)

Enviando 23 registros a AmigoCloud (Dataset 390174)...
✅ Carga masiva exitosa.
